# F1 Undercut/Overcut Breakeven Model

Pit strategy treated as pure cost-benefit, not physics. Fit tyre degradation as a regression,
get pit-lane loss from real data, then do some algebra to find the breakeven gap for an undercut.

*(Quick note: I built this without live internet, so the data-loading cells weren't actually run
when I wrote it. I tested the maths on fake data with a known answer first though, so it should
just work when you run it top to bottom.)*

In [ ]:
# %pip (not !pip) installs into the actual kernel environment
%pip install --quiet fastf1 linearmodels

## Setup

In [ ]:
import fastf1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from linearmodels.panel import PanelOLS
import warnings

warnings.filterwarnings("ignore")

fastf1.Cache.enable_cache("f1_cache")  # caches locally after first download

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## Getting the data

Using Hungary and Netherlands 2026, both tracks where overtaking is hard so pit strategy actually
decides positions rather than pace. Good for testing a breakeven model against real strategic
battles later.

Change `RACES` below to run this on a different weekend.

In [ ]:
YEAR = 2026
RACES = ["Hungary", "Netherlands"]   # hard to overtake on
SESSION_TYPE = "R"                   # Race

sessions = {}
for race in RACES:
    s = fastf1.get_session(YEAR, race, SESSION_TYPE)
    s.load(laps=True, telemetry=False, weather=False, messages=False)
    sessions[race] = s
    print(f"{race}: {len(s.laps)} laps loaded, {s.laps['Driver'].nunique()} drivers")

# stack both races, tag with Race so the fixed effects don't mix them
laps = pd.concat(
    [sessions[r].laps.copy().assign(Race=r) for r in RACES],
    ignore_index=True
)
laps.head()

## Cleaning: green-flag laps only

Dropping:
- in-laps / out-laps (pit-lane time isn't racing pace)
- anything not `TrackStatus == '1'` (SC, VSC, red/yellow flags)
- laps FastF1 flags as inaccurate
- missing lap times

Loses a decent chunk of laps but that's fine, we just want clean tyre-age signal.

In [ ]:
clean = (
    laps
    .pipe(lambda df: df[df["PitInTime"].isna() & df["PitOutTime"].isna()])
    .pipe(lambda df: df[df["TrackStatus"] == "1"])
    .pipe(lambda df: df[df["IsAccurate"] == True])
    .pipe(lambda df: df[df["LapTime"].notna()])
    .copy()
)

clean["LapTimeSeconds"] = clean["LapTime"].dt.total_seconds()
clean["TyreAge"] = clean["TyreLife"].astype(float)
clean["TyreAge2"] = clean["TyreAge"] ** 2

print(f"Raw laps:   {len(laps):>5}")
print(f"Clean laps: {len(clean):>5}  ({len(clean) / len(laps):.0%} retained)")
clean[["Race", "Driver", "LapNumber", "Compound", "TyreAge", "LapTimeSeconds"]].head()

## Degradation regression

$$
\text{LapTime}_{it} = \alpha_i + \beta_1 \, \text{TyreAge}_{it} + \beta_2 \, \text{TyreAge}_{it}^2
+ \gamma \, \text{LapNumber}_{it} + \sum_c \delta_c \, \text{Compound}_{it}^{c} + \varepsilon_{it}
$$

- **Driver fixed effects** ($\alpha_i$) - comparing each driver to themselves, not a Mercedes on
  lap 20 to a backmarker on lap 3.
- **Quadratic in tyre age** - degradation usually isn't linear, tends to be flat then falls off.
- **LapNumber control** - race gets faster over time (fuel burn, track evolution), need to
  separate that from actual tyre wear.
- **Compound dummies** - different tyres have different pace, not just different wear rates.
- **Clustered SEs by driver** - laps within a driver's stint aren't independent, plain OLS errors
  would be too small.

The cell below also auto-handles the usual real-data headaches (missing values, a driver with
barely any clean laps, a compound with no variation) instead of just crashing.

In [ ]:
MIN_LAPS_PER_DRIVER = 5  # avoids rank-deficient exog from thin entities

# drop missing/non-finite values first, or the rank/SVD check below crashes
required_cols = ["TyreAge", "TyreAge2", "LapNumber", "LapTimeSeconds", "Compound"]
n_before = len(clean)
clean_reg = clean.dropna(subset=required_cols).copy()
clean_reg = clean_reg[np.isfinite(clean_reg[["TyreAge", "TyreAge2", "LapNumber", "LapTimeSeconds"]]).all(axis=1)]
if len(clean_reg) < n_before:
    print(f"Dropped {n_before - len(clean_reg)} row(s) with missing/non-finite values in "
          f"{required_cols} before fitting.")

lap_counts = clean_reg.groupby("Driver").size()
thin_drivers = lap_counts[lap_counts < MIN_LAPS_PER_DRIVER].index.tolist()
if thin_drivers:
    print(f"Dropping {len(thin_drivers)} driver(s) with fewer than {MIN_LAPS_PER_DRIVER} "
          f"clean laps: {thin_drivers}")
    clean_reg = clean_reg[~clean_reg["Driver"].isin(thin_drivers)]

panel = clean_reg.set_index(["Driver", "LapNumber"], drop=False)

compound_dummies = pd.get_dummies(panel["Compound"], prefix="Cmp", drop_first=True).astype(float)
zero_var_cols = compound_dummies.columns[compound_dummies.var() < 1e-8]
if len(zero_var_cols):
    print(f"Dropping compound dummy(ies) with ~no variation: {list(zero_var_cols)}")
    compound_dummies = compound_dummies.drop(columns=zero_var_cols)

exog_cols = panel[["TyreAge", "TyreAge2", "LapNumber"]].astype(float)
exog = pd.concat([exog_cols, compound_dummies], axis=1)

assert np.isfinite(exog.values).all(), "exog still contains NaN/inf -- check exog.isna().sum()"

# if TyreAge/LapNumber are still collinear, drop LapNumber and refit (see 1.4)
rank = np.linalg.matrix_rank(exog.values)
if rank < exog.shape[1]:
    print(f"exog is rank-deficient ({rank} of {exog.shape[1]} cols identified). Dropping LapNumber.")
    exog = exog.drop(columns=["LapNumber"])

driver_fe_model = PanelOLS(
    panel["LapTimeSeconds"], exog, entity_effects=True, drop_absorbed=True
)
driver_fe_res = driver_fe_model.fit(cov_type="clustered", cluster_entity=True)
print(driver_fe_res.summary)

$\beta_1$ is the marginal cost of one more lap on the tyre, in seconds. With the quadratic term,
effective marginal cost at age $a$ is $\beta_1 + 2\beta_2 a$ - this is what Day 2 uses to price
staying out.

`LapNumber` should come back negative (race gets faster over time). Worth a sanity check if it
doesn't.

## Robustness check: stint fixed effects

Same idea but fixing effects at driver+stint instead of just driver. Stricter control, but it
also means `TyreAge` and `LapNumber` move in lockstep within a single stint, so `LapNumber` gets
dropped as collinear and this version can't really separate wear from track evolution. Treating
this as a shape check on the curve, not the headline number.

In [ ]:
stint_panel = clean.copy()
stint_panel["StintID"] = stint_panel["Driver"] + "_" + stint_panel["Race"] + "_S" + stint_panel["Stint"].astype(int).astype(str)
stint_panel = stint_panel.set_index(["StintID", "LapNumber"], drop=False)

exog_stint = stint_panel[["TyreAge", "TyreAge2", "LapNumber"]].astype(float)

stint_fe_model = PanelOLS(
    stint_panel["LapTimeSeconds"], exog_stint, entity_effects=True, drop_absorbed=True
)
stint_fe_res = stint_fe_model.fit(cov_type="clustered", cluster_entity=True)
print(stint_fe_res.summary)

In [ ]:
# predicted lap-time penalty vs. a fresh tyre, holding driver/compound fixed
beta_age = driver_fe_res.params["TyreAge"]
beta_age2 = driver_fe_res.params["TyreAge2"]

ages = np.arange(0, 35)
predicted_penalty = beta_age * ages + beta_age2 * ages ** 2

fig, ax = plt.subplots()
ax.plot(ages, predicted_penalty, color="crimson", lw=2, label="Fitted age penalty (driver-FE model)")
ax.set_xlabel("Tyre age (laps)")
ax.set_ylabel("Predicted lap-time penalty vs. a fresh tyre (s)")
ax.set_title("Estimated tyre degradation curve")
ax.legend()
plt.show()

print(f"Marginal cost of one more lap at tyre age  5: {beta_age + 2*beta_age2*5:.3f} s/lap")
print(f"Marginal cost of one more lap at tyre age 20: {beta_age + 2*beta_age2*20:.3f} s/lap")

## Pit-lane loss 

Not calculating this from pit lane length/speed limit, just measuring what actually happened:
in-lap and out-lap time vs. that driver's own pace in the laps just before/after the stop.

$$
\text{PitLoss} = (\text{InLapTime} - \text{LocalBaseline}_{\text{pre}})
+ (\text{OutLapTime} - \text{LocalBaseline}_{\text{post}})
$$

In [ ]:
def local_pit_loss(laps_df, window=5):
    '''Empirical pit-lane loss per stop: in/out lap time vs. local baseline pace.'''
    is_clean = (
        laps_df["PitInTime"].isna() & laps_df["PitOutTime"].isna()
        & (laps_df["TrackStatus"] == "1") & (laps_df["IsAccurate"] == True)
    )
    results = []
    for (race, drv), g in laps_df.groupby(["Race", "Driver"]):
        g = g.sort_values("LapNumber")
        in_lap = g[g["PitInTime"].notna()]
        out_lap = g[g["PitOutTime"].notna()]
        for _, in_row in in_lap.iterrows():
            in_num = in_row["LapNumber"]
            out_row = out_lap[out_lap["LapNumber"] == in_num + 1]
            if out_row.empty:
                continue
            out_num = out_row["LapNumber"].iloc[0]

            pre = g[(g["LapNumber"] >= in_num - window) & (g["LapNumber"] < in_num) & is_clean.loc[g.index]]
            post = g[(g["LapNumber"] > out_num) & (g["LapNumber"] <= out_num + window) & is_clean.loc[g.index]]
            if pre.empty or post.empty:
                continue

            in_delta = in_row["LapTime"].total_seconds() - pre["LapTime"].dt.total_seconds().median()
            out_delta = out_row["LapTime"].dt.total_seconds().iloc[0] - post["LapTime"].dt.total_seconds().median()

            results.append(dict(
                Race=race, Driver=drv, InLap=int(in_num),
                InLapDelta=in_delta, OutLapDelta=out_delta, TotalPitLoss=in_delta + out_delta
            ))
    return pd.DataFrame(results)

pit_loss_table = local_pit_loss(laps)
pit_loss_table

In [ ]:
PIT_LOSS = pit_loss_table["TotalPitLoss"].median()  # median, not mean: robust to the odd slow/scruffy stop

print(f"Empirical pit-lane loss (median across {len(pit_loss_table)} stops): {PIT_LOSS:.2f} s")
print(f"  In-lap component (median):  {pit_loss_table['InLapDelta'].median():.2f} s")
print(f"  Out-lap component (median): {pit_loss_table['OutLapDelta'].median():.2f} s")
print(f"  Spread (IQR):               {pit_loss_table['TotalPitLoss'].quantile(.75) - pit_loss_table['TotalPitLoss'].quantile(.25):.2f} s")

pit_loss_table["TotalPitLoss"].plot(kind="hist", bins=15, edgecolor="white", title="Distribution of total pit-lane loss per stop")
plt.xlabel("Seconds")
plt.show()

Pit loss isn't actually one fixed number (crew speed, traffic, release timing all vary), the IQR
below shows that. The breakeven model still uses a single point estimate though, worth keeping in
mind.

## Breakeven: marginal cost vs fixed cost

- $\beta_1, \beta_2$: marginal cost of staying out another lap (degradation)
- PitLoss: fixed cost of pitting

If a driver pits now while a rival (tyre age $t_0$, gap $G$ seconds) stays out $k$ more laps, the
advantage built up is:

$$
\text{Advantage}(t_0, k) = \sum_{j=1}^{k} \Big[\big(\beta_1(t_0+j) + \beta_2(t_0+j)^2\big) - \big(\beta_1 j + \beta_2 j^2\big)\Big]
$$

Breakeven gap, i.e. the biggest gap you can be behind and still expect to have drawn level by the
time the rival pits:

$$
G^{*}(t_0, k) = \text{Advantage}(t_0, k) - \text{PitLoss}
$$

If actual gap $G < G^*$, undercut is predicted to work.

*(Note: PitLoss only appears once here because the rival hasn't paid theirs yet at this
comparison point. Check after both have pitted instead and PitLoss roughly cancels out. The
backtest below checks at the same moment the formula describes, so it stays consistent.)*

In [ ]:
def age_penalty(age, beta1, beta2):
    '''Predicted lap-time penalty (s) from tyre age, vs. a brand-new tyre.'''
    return beta1 * age + beta2 * age ** 2


def cumulative_advantage(t0, k, beta1, beta2):
    '''Pace advantage (s) built up over a k-lap undercut window, given rival tyre age t0.'''
    j = np.arange(1, k + 1)
    rival_penalty = age_penalty(t0 + j, beta1, beta2)
    self_penalty = age_penalty(j, beta1, beta2)
    return float(np.sum(rival_penalty - self_penalty))


def breakeven_gap(t0, k, beta1, beta2, pit_loss):
    '''Largest gap (s) you can be behind and still draw level by the time the rival pits.'''
    return cumulative_advantage(t0, k, beta1, beta2) - pit_loss


def min_laps_to_breakeven(gap, t0, beta1, beta2, pit_loss, horizon=20):
    '''Smallest window k needed to close a gap. None if not closeable within `horizon` laps.'''
    for k in range(1, horizon + 1):
        if breakeven_gap(t0, k, beta1, beta2, pit_loss) >= gap:
            return k
    return None


# quick worked example
example_t0, example_k, example_gap = 12, 4, 1.5
be = breakeven_gap(example_t0, example_k, beta_age, beta_age2, PIT_LOSS)
print(f"Rival's tyre age when undercutter pits: {example_t0} laps")
print(f"Undercut window: {example_k} laps")
print(f"Breakeven gap: {be:.2f} s   (actual gap assumed: {example_gap} s)")
print("Undercut predicted to:", "SUCCEED" if example_gap < be else "FAIL")

## Does the rival cover?

Simple decision rule, not a solved game: rival should cover if staying out would let the model's
predicted undercut succeed, otherwise there's no point covering.

In [ ]:
def rival_best_response(gap, t0, planned_window, beta1, beta2, pit_loss):
    '''Simplified decision rule (not a solved game): cover, or extend?'''
    be = breakeven_gap(t0, planned_window, beta1, beta2, pit_loss)
    if gap < be:
        return "COVER", be
    return "EXTEND", be


response, be_value = rival_best_response(example_gap, example_t0, example_k, beta_age, beta_age2, PIT_LOSS)
print(f"Model-consistent rival response: {response}  (breakeven gap was {be_value:.2f}s vs. actual gap {example_gap}s)")

## Finding real undercut/overcut episodes

Instead of hand-picking "famous" battles, scanning the timing data itself for driver pairs close
together on track where one pits and the other pits a few laps later. `Time` in FastF1 is
cumulative session time, so the gap between two drivers' `Time` at the same lap is a real, direct
number.

In [ ]:
def build_time_table(laps_df):
    '''Cumulative session time (s) for each driver at each lap number.'''
    t = laps_df.pivot_table(index="LapNumber", columns="Driver", values="Time", aggfunc="first")
    return t.apply(lambda col: col.dt.total_seconds())


def find_pit_sequences(laps_df, race_name, gap_threshold=3.0, max_window=6):
    '''
    Candidate undercut/overcut episodes: driver pairs within `gap_threshold` seconds on
    track, where one pits and the other pits within `max_window` laps.

    Gap is measured the lap *before* the in-lap, since Time on the in-lap itself already
    includes some of that stop's cost.
    '''
    race_laps = laps_df[laps_df["Race"] == race_name]
    time_table = build_time_table(race_laps)
    pit_laps = race_laps.loc[race_laps["PitInTime"].notna(), ["Driver", "LapNumber"]].rename(
        columns={"LapNumber": "PitLap"}
    )

    episodes = []
    for _, row in pit_laps.iterrows():
        drv, lap = row["Driver"], row["PitLap"]
        ref_lap = lap - 1  # last clean lap before pitting
        if ref_lap not in time_table.index or drv not in time_table.columns:
            continue
        drv_time = time_table.loc[ref_lap, drv]
        if pd.isna(drv_time):
            continue

        others = time_table.loc[ref_lap].drop(labels=[drv], errors="ignore").dropna()
        close = others[(others - drv_time).abs() <= gap_threshold]

        for rival, riv_time in close.items():
            rival_pits = pit_laps[
                (pit_laps["Driver"] == rival)
                & (pit_laps["PitLap"] > lap)
                & (pit_laps["PitLap"] <= lap + max_window)
            ]
            if rival_pits.empty:
                continue

            gap = drv_time - riv_time  # positive = undercutter behind
            episodes.append(dict(
                Race=race_name, Undercutter=drv, Rival=rival,
                UndercutLap=int(lap), RivalPitLap=int(rival_pits["PitLap"].iloc[0]),
                WindowK=int(rival_pits["PitLap"].iloc[0] - lap),
                GapSeconds=round(float(gap), 3),
            ))
    return pd.DataFrame(episodes).drop_duplicates(subset=["Race", "Undercutter", "Rival"])


all_episodes = pd.concat(
    [find_pit_sequences(laps, r) for r in RACES], ignore_index=True
)
all_episodes.sort_values("GapSeconds", key=abs)

## Backtest

Checking the model's prediction against what actually happened, right before the rival makes
their own stop (same moment the formula in 2.1 describes).

In [ ]:
def evaluate_episode(ep, laps_df, beta1, beta2, pit_loss):
    '''
    Checks the prediction at the same moment the formula in 2.1 describes: the rival's
    last normal lap, just before their own in-lap. Checking any later (after the rival
    has also pitted) would test a different, pit-loss-free claim -- see 2.1.
    '''
    race_laps = laps_df[laps_df["Race"] == ep["Race"]]
    time_table = build_time_table(race_laps)

    t0_row = race_laps[(race_laps["Driver"] == ep["Rival"]) & (race_laps["LapNumber"] == ep["UndercutLap"])]
    t0 = float(t0_row["TyreLife"].iloc[0]) if not t0_row.empty else np.nan

    be_star = breakeven_gap(t0, ep["WindowK"], beta1, beta2, pit_loss) if pd.notna(t0) else np.nan
    predicted_success = bool(ep["GapSeconds"] < be_star) if pd.notna(be_star) else None

    check_lap = ep["RivalPitLap"] - 1  # rival's last lap before their own in-lap
    actual_gap = np.nan
    if check_lap in time_table.index:
        drv_t = time_table.loc[check_lap].get(ep["Undercutter"], np.nan)
        riv_t = time_table.loc[check_lap].get(ep["Rival"], np.nan)
        if pd.notna(drv_t) and pd.notna(riv_t):
            actual_gap = drv_t - riv_t

    actual_success = bool(actual_gap < 0) if pd.notna(actual_gap) else None
    correct = (predicted_success == actual_success) if predicted_success is not None and actual_success is not None else None

    return dict(
        **ep, TyreAgeAtPit=t0, BreakevenGapStar=be_star,
        PredictedSuccess=predicted_success, ActualGapBeforeRivalStop=actual_gap,
        ActualSuccess=actual_success, CorrectCall=correct,
    )


backtest = pd.DataFrame([
    evaluate_episode(ep, laps, beta_age, beta_age2, PIT_LOSS)
    for _, ep in all_episodes.iterrows()
])

backtest_valid = backtest.dropna(subset=["CorrectCall"])
print(f"Episodes with a resolvable outcome: {len(backtest_valid)} of {len(backtest)} candidates found")
if len(backtest_valid):
    print(f"Model accuracy on these episodes: {backtest_valid['CorrectCall'].mean():.0%}")

backtest_valid.sort_values("GapSeconds", key=abs).head(10)